**EXPLORING DATASETS**

Analysing the datasets provided to find insights like unique labels, values and categories, etc.., and also to find anomaly values and null rows and columns.

This helps us to prepare the dataset for the processing step where the outlier data, missing data are processed to remove ambuigity, error and bias in the data.

In [22]:
#IMPORTS

import pandas as pd
from pathlib import Path as pl

#DIRECTORY WORK

RAW_DIR = pl("../data/raw")

datasets = {}

for file in RAW_DIR.glob("*.csv"):
    df = pd.read_csv(file)
    datasets[file.stem] = df

print(f"Loaded {len(datasets)} datasets from {str(RAW_DIR).replace('..', '')}")

#FUND MASTER EXPLORE

fund_master = datasets["01_fund_master"]

print("\nUnique Fund Houses:")
print(fund_master["fund_house"].unique())

print("\nCategories:")
print(fund_master["category"].unique())

print("\nSub Categories:")
print(fund_master["sub_category"].unique())

print("\nRisk Grades:")
print(fund_master["risk_category"].unique())

#NAV HISTORY

nav = datasets["02_nav_history"]

nav["date"] = pd.to_datetime(nav["date"])

print("\nDate Range:")
print(nav["date"].min(), "to", nav["date"].max())

print("\nNegative NAV Values:")
print((nav["nav"] <= 0).sum())

#AMFI VALIDATION IN HISTORY

master_codes = set(
    fund_master["amfi_code"].astype(str)
)

nav_codes = set(
    nav["amfi_code"].astype(str)
)

missing_codes = master_codes - nav_codes

print(f"Fund Master Codes: {len(master_codes)}")
print(f"NAV History Codes: {len(nav_codes)}")

print(f"\nMissing Codes: {len(missing_codes)}")

if missing_codes:
    print(missing_codes)
else:
    print("All AMFI codes exist in NAV history.")

#ANOMALY DETECTION

print("\n" + "="*80)
print("ANOMALY DETECTION REPORT")
print("="*80)

for name, df in datasets.items():

    print(f"\n{name}")
    print("-" * 50)

    # Missing Values
    missing = df.isnull().sum().sum()

    if missing > 0:
        print(f"Missing Values Found: {missing}")
        print(df.isnull().sum()[df.isnull().sum() > 0])

    # Duplicate Rows
    duplicates = df.duplicated().sum()

    if duplicates > 0:
        print(f"Duplicate Rows Found: {duplicates}")
    else:
        print("No Duplicate Rows Found.")

    # Columns that must always contain positive values

    positive_checks = {
    "01_fund_master": [
        "expense_ratio_pct",
        "min_sip_amount",
        "min_lumpsum_amount"
    ],

    "02_nav_history": [
        "nav"
    ],

    "03_aum_by_fund_house": [
        "aum_crore"
    ],

    "04_monthly_sip_inflows": [
        "sip_inflow_crore"
    ],

    "06_industry_folio_count": [
        "folio_count"
    ],

    "08_investor_transactions": [
        "transaction_amount"
    ],

    "09_portfolio_holdings": [
        "weight_pct"
    ],

    "10_benchmark_indices": [
        "index_value"
    ]
}


    if name in positive_checks:

      for col in positive_checks[name]:

        if col in df.columns:

            invalid = (df[col] <= 0).sum()

            if invalid > 0:
                print(
                    f"{col}: {invalid} values are <= 0 "
                    "(expected positive values)"
                )
            else:
                print(
                    f"{col}: All values are positive"
                )

Loaded 10 datasets from \data\raw

Unique Fund Houses:
<StringArray>
[         'SBI Mutual Fund',         'HDFC Mutual Fund',
      'ICICI Prudential MF',          'Nippon India MF',
        'Kotak Mahindra MF',         'Axis Mutual Fund',
 'Aditya Birla Sun Life MF',          'UTI Mutual Fund',
           'Mirae Asset MF',          'DSP Mutual Fund']
Length: 10, dtype: str

Categories:
<StringArray>
['Equity', 'Debt']
Length: 2, dtype: str

Sub Categories:
<StringArray>
[      'Large Cap',       'Small Cap',            'Gilt',         'Mid Cap',
  'Short Duration',           'Value',          'Liquid',       'Index/ETF',
       'Flexi Cap',           'Index', 'Large & Mid Cap',            'ELSS']
Length: 12, dtype: str

Risk Grades:
<StringArray>
['Moderate', 'Very High', 'Low', 'High', 'Moderately High']
Length: 5, dtype: str

Date Range:
2022-01-03 00:00:00 to 2026-05-29 00:00:00

Negative NAV Values:
0
Fund Master Codes: 40
NAV History Codes: 40

Missing Codes: 0
All AMFI codes exi

**Anomaly Report**
